# Customer Data Processing & RFM Analysis using Apache Spark

**Pipeline:** Raw CSV → Cleaning (PySpark) → RFM feature engineering → Scoring → Segmentation → CSV export → Power BI dashboard

This notebook shows the **data-processing stage** of the project. All numbers below are computed from the actual dataset with PySpark; nothing is hard-coded.

## 1. Install & Import
Install PySpark (pinned version, works on Colab and local Jupyter), then start a `SparkSession`.

In [ ]:
!pip install -q pyspark==3.5.3

In [ ]:
import glob, os, shutil
from IPython.display import display
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("RFM_Customer_Segmentation")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Apache Spark version:", spark.version)

def preview(df, n=5):
    """Show the first n rows of a Spark DataFrame as a neat table."""
    return df.limit(n).toPandas()

## 2. Load Raw Data
Set the path to your dataset in **`DATA_PATH`** (the only line you may need to change).
In Google Colab: upload the CSV from the left *Files* panel, then use `/content/<filename>.csv`.

In [ ]:
# >>> CHANGE THIS to the location of your CSV file <<<
DATA_PATH = "online_retail.csv"

In [ ]:
raw = (spark.read
       .option("header", True)
       .option("inferSchema", True)
       .option("encoding", "ISO-8859-1")      # Online Retail CSV is not UTF-8
       .csv(DATA_PATH))

# Standardise column names if a different version of the dataset is used
raw = raw.select([F.col(c).alias({"Invoice": "InvoiceNo", "Price": "UnitPrice",
                                  "Customer ID": "CustomerID"}.get(c, c)) for c in raw.columns])

required = ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
missing_cols = [c for c in required if c not in raw.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}"

raw_count = raw.count()
print(f"Rows: {raw_count:,}   |   Columns: {len(raw.columns)}")
raw.printSchema()
preview(raw, 5)

## 3. Data Cleaning using PySpark
Steps: check missing values and duplicates → remove duplicates → drop missing Customer IDs → convert data types → remove invalid transactions (cancelled invoices, non-positive quantity or price).

In [ ]:
# 3.1 Missing values per column (null or blank)
missing = raw.select([F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
                      for c in raw.columns])
print("Missing values per column:")
display(missing.toPandas())

# 3.2 Exact duplicate rows
dup_count = raw_count - raw.dropDuplicates().count()
print(f"Exact duplicate rows: {dup_count:,}")

In [ ]:
# Set to False to keep duplicate rows (e.g. to reproduce a pipeline that did not remove them)
DROP_DUPLICATES = True

steps = [("Raw data", raw_count)]

df = raw.dropDuplicates() if DROP_DUPLICATES else raw
steps.append(("After removing duplicates", df.count()))

# Missing / blank Customer IDs
df = df.filter(F.col("CustomerID").isNotNull() & (F.trim(F.col("CustomerID").cast("string")) != ""))
steps.append(("After removing missing CustomerID", df.count()))

# Type conversion (CustomerID 12583.0 -> 12583; text dates -> timestamp)
df = (df
      .withColumn("InvoiceNo",  F.col("InvoiceNo").cast("string"))
      .withColumn("Quantity",   F.col("Quantity").cast("int"))
      .withColumn("UnitPrice",  F.col("UnitPrice").cast("double"))
      .withColumn("CustomerID", F.col("CustomerID").cast("double").cast("long"))
      .withColumn("InvoiceDate", F.coalesce(
            F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"),
            F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm:ss"),
            F.to_timestamp("InvoiceDate", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("InvoiceDate", "yyyy-MM-dd H:mm"),
            F.to_timestamp("InvoiceDate", "d-M-yyyy H:mm")))
      .filter(F.col("CustomerID").isNotNull() & F.col("InvoiceDate").isNotNull()))
steps.append(("After type conversion (valid IDs & dates)", df.count()))

# Invalid transactions: cancelled invoices (start with 'C'), Quantity <= 0, UnitPrice <= 0
df = df.filter(~F.col("InvoiceNo").startswith("C") & (F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
df = df.withColumn("TotalPrice", F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
steps.append(("After removing invalid transactions", df.count()))

clean = df.cache()
print(f"BEFORE cleaning: {raw_count:,} rows   ->   AFTER cleaning: {clean.count():,} rows\n")
for name, n in steps:
    print(f"{name:<45}{n:>12,}")

## 4. Basic Processing Output
Quick summary of the cleaned dataset (computed with Spark aggregations).

In [ ]:
print("Cleaned data preview:")
display(preview(clean, 5))

summary = clean.agg(
    F.countDistinct("CustomerID").alias("Unique_Customers"),
    F.countDistinct("InvoiceNo").alias("Total_Transactions (invoices)"),
    F.count("*").alias("Total_Line_Items"),
    F.min("InvoiceDate").alias("First_Date"),
    F.max("InvoiceDate").alias("Last_Date"),
    F.round(F.sum("TotalPrice"), 2).alias("Total_Revenue"))
summary.toPandas().T.rename(columns={0: "Value"})

## 5. RFM Feature Engineering (PySpark)
One row per customer, using `groupBy` + aggregations:
- **Recency** – days since the customer's last purchase (reference date = last date in the data + 1 day)
- **Frequency** – number of distinct invoices
- **Monetary** – total amount spent

In [ ]:
ref_date = clean.agg(F.date_add(F.to_date(F.max("InvoiceDate")), 1)).collect()[0][0]
print("Reference date:", ref_date)

rfm = (clean.groupBy("CustomerID")
       .agg(F.first("Country").alias("Country"),
            F.datediff(F.lit(ref_date), F.to_date(F.max("InvoiceDate"))).alias("Recency"),
            F.countDistinct("InvoiceNo").alias("Frequency"),
            F.round(F.sum("TotalPrice"), 2).alias("Monetary")))

print("Customers:", rfm.count())
preview(rfm.orderBy("CustomerID"), 5)

## 6. RFM Scoring (scores 1–5, 5 = best)
- **R score:** `NTILE(5)` window function on Recency (more recent → higher score)
- **M score:** `NTILE(5)` window function on Monetary (higher spend → higher score)
- **F score:** fixed bins (1 order = 1, 2 = 2, 3 = 3, 4–6 = 4, 7+ = 5), because many customers share the same order count and quintiles would split identical customers into different groups

Overall score: `RFM_Score` = R, F and M joined as text (e.g. `545`), and `RFM_Total` = R + F + M.

In [ ]:
scored = (rfm
    .withColumn("R_Score", F.ntile(5).over(Window.orderBy(F.col("Recency").desc(), F.col("CustomerID"))))
    .withColumn("F_Score", F.when(F.col("Frequency") == 1, 1)
                            .when(F.col("Frequency") == 2, 2)
                            .when(F.col("Frequency") == 3, 3)
                            .when(F.col("Frequency") <= 6, 4)
                            .otherwise(5))
    .withColumn("M_Score", F.ntile(5).over(Window.orderBy(F.col("Monetary").asc(), F.col("CustomerID"))))
    .withColumn("RFM_Score", F.concat_ws("", "R_Score", "F_Score", "M_Score"))
    .withColumn("RFM_Total", F.col("R_Score") + F.col("F_Score") + F.col("M_Score")))

preview(scored.orderBy("CustomerID"), 5)

## 7. Customer Segmentation
Same rules as used for the Power BI dashboard:

| Segment | Rule |
|---|---|
| Champions | R ≥ 4 and F ≥ 4 and M ≥ 4 |
| Loyal Customers | R ≥ 3 and F ≥ 3 (not Champions) |
| Potential Loyalists | R ≥ 3 and F < 3 |
| At Risk | R < 3 and F ≥ 3 |
| Lost | R < 3 and F < 3 |

In [ ]:
final = scored.withColumn("Segment",
    F.when((F.col("R_Score") >= 4) & (F.col("F_Score") >= 4) & (F.col("M_Score") >= 4), "Champions")
     .when((F.col("R_Score") >= 3) & (F.col("F_Score") >= 3), "Loyal Customers")
     .when((F.col("R_Score") >= 3) & (F.col("F_Score") < 3), "Potential Loyalists")
     .when((F.col("R_Score") < 3) & (F.col("F_Score") >= 3), "At Risk")
     .otherwise("Lost"))

segment_summary = (final.groupBy("Segment")
    .agg(F.count("*").alias("Customers"),
         F.round(F.sum("Monetary"), 2).alias("TotalRevenue"),
         F.round(F.avg("Monetary"), 2).alias("AvgSpend"),
         F.round(F.avg("Frequency"), 2).alias("AvgOrders"),
         F.round(F.avg("Recency"), 1).alias("AvgRecencyDays"))
    .orderBy(F.desc("Customers")))
segment_summary.toPandas()

## 8. Final Processed Dataset (first 10 rows)

In [ ]:
final_cols = ["CustomerID", "Country", "Recency", "Frequency", "Monetary",
              "R_Score", "F_Score", "M_Score", "RFM_Score", "Segment"]
final_out = final.select(final_cols)
preview(final_out.orderBy("CustomerID"), 10)

## 9. Export for Power BI
Spark writes CSVs as a folder of part-files; `coalesce(1)` + a rename gives a single file per table.
`rfm_segments.csv` is the final customer-level dataset. The three small summary tables used by the dashboard are exported the same way.

In [ ]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

monthly_revenue = (clean.groupBy(F.date_format("InvoiceDate", "yyyy-MM").alias("Month"))
    .agg(F.round(F.sum("TotalPrice"), 2).alias("Revenue"),
         F.countDistinct("InvoiceNo").alias("Orders"),
         F.countDistinct("CustomerID").alias("Customers")).orderBy("Month"))

revenue_by_country = (clean.groupBy("Country")
    .agg(F.round(F.sum("TotalPrice"), 2).alias("Revenue"),
         F.countDistinct("CustomerID").alias("Customers"),
         F.countDistinct("InvoiceNo").alias("Orders")).orderBy(F.desc("Revenue")))

def export_csv(sdf, name):
    tmp = os.path.join(OUTPUT_DIR, f"_{name}")
    sdf.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp)
    part = glob.glob(os.path.join(tmp, "part-*.csv"))[0]
    target = os.path.join(OUTPUT_DIR, f"{name}.csv")
    shutil.move(part, target)
    shutil.rmtree(tmp)
    print(f"Saved {target}  ({sdf.count():,} rows)")

export_csv(final_out, "rfm_segments")
export_csv(segment_summary, "segment_summary")
export_csv(monthly_revenue, "monthly_revenue")
export_csv(revenue_by_country, "revenue_by_country")

In [ ]:
# Google Colab only: download the files for Power BI (ignored elsewhere)
try:
    from google.colab import files
    for f in ["rfm_segments", "segment_summary", "monthly_revenue", "revenue_by_country"]:
        files.download(os.path.join(OUTPUT_DIR, f + ".csv"))
except ImportError:
    print("Not running in Colab - files are in the 'output' folder.")

## Conclusion
Apache Spark was used for scalable data loading, cleaning, transformation, customer-level aggregation, RFM feature engineering and segmentation. The processed dataset was then used for visualization and dashboarding in Power BI.